## Initialisation

This notebook profiles how Rescue planner runtime changes with horizon and number of communities. Run the notebook from top to bottom after generating new experiments.

The main stages are:

1. Build uniquely named experiment configurations.
2. Run the planner and extract one row per repetition into **df**.
3. Clean numeric timing fields into **plot_df**.
4. Aggregate repetitions into **average_times**.
5. Plot total/component timings and optionally compare regression models.


In [1]:
from scripts.AbstractExperiments import ExperimentRunner, GenerateConfigs
from EnvironmentBuilder.SearchRescue.Rescue import Rescue
import EnvironmentBuilder.SearchRescue.Drawing as SearchRescueDraw
from scripts.TexTables import SaveDataFrameToTexTemplate
from copy import deepcopy
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plt.rcParams.update({
    "font.family": "Charter",
    "font.size": 16,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})
def latex_config_name(name):
    return "$" + name.replace("^0", "^{0}") + "$"

In [2]:
def GenerateTeamConfigs(teams_:list, name_prefix=None):
    configs = {}
    if name_prefix is None:
        name_prefix = f"{len(teams_)}_"
    # Act-Utilitarianism
    con = {"Theories": [["Add_Util", "Utility", 0]], "Considerations": []}
    for t in teams_:
        con["Considerations"].append([f"{t}:wellbeing", "Add_Util"])
    configs[f"{name_prefix}add"] = con

    # Balance
    con = {"Theories": [], "Considerations": []}
    for t in teams_:
        con["Theories"].append([f"{t}", "Utility", 0])
        con["Considerations"].append([f"{t}:wellbeing", f"{t}"])
    configs[f"{name_prefix}bal"] = con

    # Fairness
    con = {"Theories": [["Fair", "Fairness", 0]], "Considerations": []}
    for t in teams_:
        con["Considerations"].append([f"{t}:wellbeing", ["Fair"]])
    configs[f"{name_prefix}fair"] = con

    # Fairness + Others
    con = {"Theories": [["Fair", "Fairness", 0]], "Considerations": []}
    for t in teams_:
        con["Theories"].append([f"{t}", "Utility", 0])
        con["Considerations"].append([f"{t}:wellbeing", [f"{t}", "Fair"]])
    configs[f"{name_prefix}fair_bal"] = con

    # Fairness + Act-Utilitarianism
    con = {
        "Theories": [
            ["Fair", "Fairness", 0],
            ["Add_Util", "Utility", 0],
        ],
        "Considerations": [],
    }
    for t in teams_:
        con["Considerations"].append(
            [f"{t}:wellbeing", ["Fair", "Add_Util"]]
        )
    # Include horizon/community information so later horizons do not overwrite
    # earlier Fair + AU experiment files.
    configs[f"{name_prefix}fair_AU"] = con

    # Rawls
    con = {"Theories":[["Rawls", "Maximin", 0]], "Considerations": []}
    for t in teams_:
        con["Considerations"].append([f"{t}:wellbeing", ["Rawls"]])
    configs[f"{name_prefix}rawl"] = con

    # Rawls + Others
    con = {"Theories": [["Rawls", "Maximin", 0]], "Considerations": []}
    for t in teams_:
        con["Theories"].append([f"{t}", "Utility", 0])
        con["Considerations"].append([f"{t}:wellbeing", [f"{t}", "Rawls"]])
    configs[f"{name_prefix}rawl_bal"] = con

    # Rawls + AU
    con = {"Theories": [["Rawls", "Maximin", 0], ["Add_Util", "Utility", 0]], "Considerations": []}
    for t in teams_:
        con["Theories"].append([f"{t}", "Utility", 0])
        con["Considerations"].append([f"{t}:wellbeing", ["Rawls", "Add_Util"]])
    configs[f"{name_prefix}rawl_AU"] = con

    return configs

def MakeTeamWeights(teams_):
    p = [2**i for i in range(0, len(teams_))]
    total = sum(p)
    w = [i / total for i in p]
    o = {}
    for t in teams_:
        for i, t in enumerate(teams_):
            o[f"{t}_hosp_success"] = w[i]
    return o

markers = {
    "Separate Theories": "o",
    "Act-Utilitarianism": "s",
    "Fairness": "^",
    "Rawls": "*",
    "Fair + balance": ".",
    "Rawls + balance": "x"
}
colours = {
    "Separate Theories": "blue",
    "Act-Utilitarianism": "black",
    "Fairness": "green",
    "Rawls": "red",
    "Fair + balance": "cyan",
    "Rawls + balance": "magenta",
}


## Build the environments

Each entry in **configs** describes one planner configuration for a particular horizon and community count. Configuration names must include both values because the experiment runner uses the name when writing files.

The corrected **Fair + AU** configuration now has distinct names at every horizon and contains both the Fairness and Additive Utility theories. Previous runs used the same name at every horizon, so horizons 3–5 were overwritten by horizon 6.


In [3]:
configs = []
env_reps = 10
for horizon_ in range(3,7):
    for Comms in range(2,7):
        teams = [f"{i}" for i in range(Comms)]
        odds = MakeTeamWeights(teams)
        configs += GenerateConfigs(inputConfigs=GenerateTeamConfigs(teams, name_prefix=f"{len(teams)}_h{horizon_}"), 
            defaultConfig={"Horizon": horizon_, "Teams":teams, "unknown_depth": 0, "Odds":odds})

# Output filenames are based on Name, so duplicates would overwrite results.
configuration_names = pd.Series([config["Name"] for config in configs])
duplicate_configuration_names = sorted(
    configuration_names[configuration_names.duplicated()].unique()
)
if duplicate_configuration_names:
    raise ValueError(
        "Configuration names must be unique; duplicates: "
        f"{duplicate_configuration_names}"
    )

er = ExperimentRunner("Rescue", configs, MoralPlanner_Location="/../../", outFolder="ProfileRescue")



## Run the planner

In [4]:
d = er.buildEnvironments()
er.runPlanner(envRepetitions=env_reps)

Written Rescue MMMDP with 13 states and theories [{'Name': 'Add_Util', 'Type': 'Utility', 'Rank': 0}].
PATH  /home/psiko/Programming/MoralPlanner/Notebooks/SearchRescue/../../Data/Experiments/Rescue/ProfileRescue/mdps/2_h3add_con0.json

Written Rescue MMMDP with 13 states and theories [{'Name': '0', 'Type': 'Utility', 'Rank': 0}, {'Name': '1', 'Type': 'Utility', 'Rank': 0}].
PATH  /home/psiko/Programming/MoralPlanner/Notebooks/SearchRescue/../../Data/Experiments/Rescue/ProfileRescue/mdps/2_h3bal_con0.json

Written Rescue MMMDP with 13 states and theories [{'Name': 'Fair', 'Type': 'Fairness', 'Rank': 0}].
PATH  /home/psiko/Programming/MoralPlanner/Notebooks/SearchRescue/../../Data/Experiments/Rescue/ProfileRescue/mdps/2_h3fair_con0.json

Written Rescue MMMDP with 13 states and theories [{'Name': 'Fair', 'Type': 'Fairness', 'Rank': 0}, {'Name': '0', 'Type': 'Utility', 'Rank': 0}, {'Name': '1', 'Type': 'Utility', 'Rank': 0}].
PATH  /home/psiko/Programming/MoralPlanner/Notebooks/SearchResc

## Extract and label raw results

**df** contains one row per planner repetition. Important columns include:

| Column | Meaning |
|---|---|
| Config_name | Human-readable moral configuration |
| Env_rep | Repetition number |
| Horizon | Planning horizon |
| communities | Number of rescue communities |
| Total_time | Complete planner runtime |
| Heuristic_time | Heuristic construction runtime |
| Plan_time | Planning/search runtime |
| Sol_time | Solution extraction runtime |
| Mehr_time | MEHR runtime |

The extraction cell also removes unused profiling fields and converts internal configuration names into labels used by the graphs.


In [4]:
er.extractData(1, env_reps)
df = pd.DataFrame(er.data)
df = df.drop(labels=['Conf_rep', 'Theories', 'Considerations', 'CQ1_time', 'CQ2_time', 'Out_time', 'Sol_reduce_time'], axis=1)
df = df.replace(["", "N/A", "NA", "nan", "None"], np.nan)
df['communities'] = df['Config_name'].str[:1]
df["Config_name"] = np.select(
    [
        df["Config_name"].str.contains("fair_bal", case=False, na=False),
        df["Config_name"].str.contains("fair_AU", case=False, na=False),
        df["Config_name"].str.contains("rawl_bal", case=False, na=False),
        df["Config_name"].str.contains("rawl_AU", case=False, na=False),
        df["Config_name"].str.contains("add", case=False, na=False),
        df["Config_name"].str.contains("bal", case=False, na=False),
        df["Config_name"].str.contains("fair", case=False, na=False),
        df["Config_name"].str.contains("rawl", case=False, na=False),
    ],
    [   
        "Fair + Separate",
        "Fair + AU",
        "Rawls + Separate",
        "Rawls + AU",
        "Act-Utilitarianism",
        "Separate Theories",
        "Fairness",
        "Rawls",
    ],
    default=df["Config_name"]
)
df.head(500)

FileNotFoundError: [Errno 2] No such file or directory: '/home/psiko/Programming/MoralPlanner/Notebooks/SearchRescue/../../Data/Experiments/Rescue/ProfileRescue/raw/2_h3fair_AU_con0_rep0.json'

### Check experiment coverage

**coverage_by_configuration** counts extracted repetitions for every configuration/horizon pair. A complete run should normally show **env_reps** observations in every populated cell.

After this correction, rerun **Build the environments**, **Run the planner**, and the extraction cell. Old files cannot recover Fair + AU horizons 3–5 because those files were overwritten during the original run.


In [ ]:
coverage_by_configuration = (
    df
    .groupby(["Config_name", "communities", "Horizon"])
    .size()
    .unstack("Horizon", fill_value=0)
    .sort_index()
)
coverage_by_configuration


## Prepare numeric timing data

**plot_df** is a cleaned copy of **df** used for analysis. It:

- converts horizon, community count, repetition, and timing columns to numbers;
- removes rows missing the core plotting fields;
- adds **Horizon_x_communities** as a simple combined problem-size measure.

**TIMING_COLUMNS** is the authoritative list of total and component timing fields accepted by the plotting helpers.


In [ ]:
plot_df = df.copy()

# Total runtime and the same component timings used by the Lost Insulin analysis.
TIMING_COLUMNS = [
    "Total_time",
    "Heuristic_time",
    "Plan_time",
    "Sol_time",
    "Mehr_time",
]
TIMING_LABELS = {
    "Total_time": "Total",
    "Heuristic_time": "Heuristic",
    "Plan_time": "Planning",
    "Sol_time": "Solution extraction",
    "Mehr_time": "MEHR",
}

missing_timing_columns = [
    column for column in TIMING_COLUMNS if column not in plot_df.columns
]
if missing_timing_columns:
    raise KeyError(f"Missing timing columns: {missing_timing_columns}")

numeric_columns = [
    "Env_rep",
    "Horizon",
    "communities",
] + TIMING_COLUMNS
for column in numeric_columns:
    plot_df[column] = pd.to_numeric(plot_df[column], errors="coerce")

plot_df = plot_df.dropna(
    subset=[
        "Config_name",
        "Horizon",
        "communities",
        "Total_time",
    ]
)
# Combined problem-size measure.
plot_df["Horizon_x_communities"] = (
    plot_df["Horizon"] * plot_df["communities"]
)
plot_df


## Aggregate repetitions

**average_times** has one row for each combination of configuration, horizon, and community count. For every timing component it contains:

- average, minimum, and maximum runtime;
- sample standard deviation;
- number of repetitions.

For example, **Average_plan_time** is the mean **Plan_time** across matching rows in **plot_df**. Most plotting functions use **average_times**, not the raw **df**.


In [ ]:
group_columns = [
    "Config_name",
    "Horizon",
    "communities",
    "Horizon_x_communities",
]
statistic_prefixes = {
    "mean": "Average",
    "min": "Minimum",
    "max": "Maximum",
    "std": "Standard_deviation",
    "size": "Number_of_repetitions",
}

average_times = (
    plot_df
    .groupby(group_columns)[TIMING_COLUMNS]
    .agg(list(statistic_prefixes))
    .reset_index()
)

# Flatten the aggregations two-level column names while retaining the old
# Average_total_time name used by the plots below.
average_times.columns = [
    column
    if statistic == ""
    else (
        f"{statistic_prefixes[statistic]}_"
        f"{column[0].lower()}{column[1:]}"
    )
    for column, statistic in average_times.columns
]

# A single repetition has no sample standard deviation.
standard_deviation_columns = [
    column for column in average_times
    if column.startswith("Standard_deviation_")
]
average_times[standard_deviation_columns] = (
    average_times[standard_deviation_columns].fillna(0)
)
average_times


## Plotting and regression helpers

**plot_average_growth** plots one timing field for each configuration. Its x-axis ticks are evenly spaced by default; set **ticks_for_existing_values_only=True** only when ticks should appear exclusively at observed values.

Set **time_column** to Total_time, Heuristic_time, Plan_time, Sol_time, or Mehr_time.

Regression options:

- **fit_models="linear"** fits \(y = a + bx\).
- **fit_models="exponential"** fits \(y = ae^{bx}\).
- **fit_models=("linear", "exponential")** overlays and compares both.
- **show_fit_metrics=True** displays the model forms, fitted coefficients, MSE, and \(R^2\).
- **return_fit_results=True** returns the metrics as a DataFrame for further analysis.

MSE is evaluated in the original runtime units. When both models are requested, they use the same positive observations so their MSE values are comparable.


In [ ]:
def timing_summary_column(time_column, statistic="Average"):
    """Return the summary-column name produced in the previous cell."""
    return f"{statistic}_{time_column[0].lower()}{time_column[1:]}"


def _normalise_fit_models(fit_models):
    if fit_models is None:
        return []
    if isinstance(fit_models, str):
        fit_models = [fit_models]

    models = [model.lower() for model in fit_models]
    unknown_models = set(models) - {"linear", "exponential"}
    if unknown_models:
        raise ValueError(
            "fit_models must contain only linear and/or exponential; "
            f"received {sorted(unknown_models)}"
        )
    return list(dict.fromkeys(models))


def _fit_growth_series(x, y, fit_models):
    """Fit models to one averaged series; MSE and R-squared use raw y units."""
    models = _normalise_fit_models(fit_models)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if "exponential" in models:
        # Use the same positive observations for both models, making their
        # original-scale MSE values directly comparable.
        valid &= y > 0
    x_valid = x[valid]
    y_valid = y[valid]

    results = []
    for model in models:
        if len(x_valid) < 2 or np.unique(x_valid).size < 2:
            continue

        if model == "linear":
            slope, intercept = np.polyfit(x_valid, y_valid, deg=1)
            scale = intercept
            predictions = intercept + slope * x_valid
            equation = f"y = {intercept:.4g} + {slope:.4g}x"
        else:
            slope, log_scale = np.polyfit(
                x_valid,
                np.log(y_valid),
                deg=1,
            )
            scale = np.exp(log_scale)
            predictions = scale * np.exp(slope * x_valid)
            equation = f"y = {scale:.4g} exp({slope:.4g}x)"

        residuals = y_valid - predictions
        mse = np.mean(residuals ** 2)
        total_sum_of_squares = np.sum(
            (y_valid - y_valid.mean()) ** 2
        )
        r_squared = (
            np.nan
            if np.isclose(total_sum_of_squares, 0)
            else 1 - np.sum(residuals ** 2) / total_sum_of_squares
        )

        results.append({
            "Model": model,
            "Coefficient_a": scale,
            "Coefficient_b": slope,
            "Equation": equation,
            "MSE": mse,
            "R_squared": r_squared,
            "Number_of_points": len(x_valid),
        })

    return results


def fit_growth_models(
    data,
    x_column,
    time_column="Total_time",
    configurations=None,
    fit_models=("linear", "exponential"),
):
    """Compare growth models for every configuration and return their metrics."""
    average_column = timing_summary_column(time_column)
    if average_column not in data:
        raise KeyError(
            f"{average_column} is unavailable. Re-run the timing summary cell."
        )

    fit_data = data.copy()
    if configurations is not None:
        fit_data = fit_data[
            fit_data["Config_name"].isin(configurations)
        ]

    records = []
    for configuration, configuration_data in fit_data.groupby(
        "Config_name",
        sort=False,
    ):
        line_data = (
            configuration_data
            .groupby(x_column, as_index=False)[average_column]
            .mean()
            .sort_values(x_column)
        )
        series_results = _fit_growth_series(
            line_data[x_column],
            line_data[average_column],
            fit_models,
        )
        for result in series_results:
            records.append({
                "Config_name": configuration,
                "Time_column": time_column,
                **result,
            })

    result_columns = [
        "Config_name",
        "Time_column",
        "Model",
        "Coefficient_a",
        "Coefficient_b",
        "Equation",
        "MSE",
        "R_squared",
        "Number_of_points",
    ]
    return pd.DataFrame(records, columns=result_columns)


def plot_average_growth(
    data,
    x_column,
    x_label,
    title,
    configurations=None,
    use_log_scale=True,
    ticks_for_existing_values_only=False,
    no_legend=False,
    ax=None,
    time_column="Total_time",
    fit_models=None,
    show_fit_metrics=False,
    return_fit_results=False,
):
    """Plot a timing metric, optionally overlaying linear/exponential fits."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
        standalone_plot = True
    else:
        fig = ax.figure
        standalone_plot = False

    average_column = timing_summary_column(time_column)
    if average_column not in data:
        raise KeyError(
            f"{average_column} is unavailable. Re-run the timing summary cell."
        )

    models = _normalise_fit_models(fit_models)
    plot_data = data.copy()
    if configurations is not None:
        plot_data = plot_data[
            plot_data["Config_name"].isin(configurations)
        ]

    all_fit_results = []
    for configuration_index, (
        configuration,
        configuration_data,
    ) in enumerate(plot_data.groupby("Config_name", sort=False)):
        line_data = (
            configuration_data
            .groupby(x_column, as_index=False)[average_column]
            .mean()
            .sort_values(x_column)
        )
        observed_line, = ax.plot(
            line_data[x_column],
            line_data[average_column],
            marker="o",
            linewidth=2,
            label=configuration,
        )

        series_results = _fit_growth_series(
            line_data[x_column],
            line_data[average_column],
            models,
        )
        all_fit_results.extend([
            {
                "Config_name": configuration,
                "Time_column": time_column,
                **result,
            }
            for result in series_results
        ])

        if series_results:
            regression_x = np.linspace(
                line_data[x_column].min(),
                line_data[x_column].max(),
                200,
            )
            for result in series_results:
                if result["Model"] == "linear":
                    regression_y = (
                        result["Coefficient_a"]
                        + result["Coefficient_b"] * regression_x
                    )
                    linestyle = "--"
                else:
                    regression_y = (
                        result["Coefficient_a"]
                        * np.exp(result["Coefficient_b"] * regression_x)
                    )
                    linestyle = ":"

                visible = np.isfinite(regression_y)
                if use_log_scale:
                    visible &= regression_y > 0

                # Add each fit-model style to the legend only once.
                fit_label = (
                    result["Model"].title() + " fit"
                    if configuration_index == 0
                    else "__nolegend__"
                )
                ax.plot(
                    regression_x[visible],
                    regression_y[visible],
                    color=observed_line.get_color(),
                    linestyle=linestyle,
                    linewidth=2,
                    alpha=0.9,
                    label=fit_label,
                )

    timing_label = TIMING_LABELS.get(
        time_column,
        time_column.replace("_", " "),
    )
    y_label = f"Average {timing_label.lower()} time"
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

    if use_log_scale:
        ax.set_yscale("log")
        ax.set_ylabel(f"{y_label} (log scale)")

    x_values = plot_data[x_column].dropna().to_numpy(dtype=float)
    if ticks_for_existing_values_only:
        ax.set_xticks(sorted(np.unique(x_values)))
    elif x_values.size:
        # A numeric locator gives evenly spaced ticks even when some x values
        # have no observation.
        integer_ticks = np.allclose(x_values, np.round(x_values))
        ax.xaxis.set_major_locator(
            MaxNLocator(nbins=8, integer=integer_ticks)
        )

    if not no_legend:
        ax.legend(
            title="Configuration / fit",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
        )

    fit_results = pd.DataFrame(all_fit_results)
    if show_fit_metrics and not fit_results.empty:
        equation_forms = {
            "linear": r"$y = a + bx$",
            "exponential": r"$y = ae^{bx}$",
        }
        metric_lines = ["Fit equations"]
        for model in models:
            metric_lines.append(
                f"{model.title()}: {equation_forms[model]}"
            )

        for configuration, configuration_results in fit_results.groupby(
            "Config_name",
            sort=False,
        ):
            metric_lines.extend(["", str(configuration)])
            for row in configuration_results.itertuples():
                metric_lines.append(
                    f"  {row.Model.title()}: "
                    f"a={row.Coefficient_a:.4g}, "
                    f"b={row.Coefficient_b:.4g}"
                )
                metric_lines.append(
                    f"    MSE={row.MSE:.4g}, "
                    f"R²={row.R_squared:.4g}"
                )

        ax.text(
            1.02,
            0.58,
            "\n".join(metric_lines),
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=12,
            linespacing=1.25,
            bbox={
                "boxstyle": "round,pad=0.5",
                "facecolor": "white",
                "edgecolor": "0.7",
                "alpha": 0.95,
            },
        )

    if standalone_plot:
        fig.tight_layout()
        plt.show()

    if return_fit_results:
        return ax, fit_results
    return ax


def plot_timing_components(
    data,
    x_column,
    x_label,
    configurations=None,
    time_columns=None,
    fit_models=None,
    use_log_scale=True,
    figsize=(14, 10),
):
    """Draw a reusable panel for the individual components of total time."""
    if time_columns is None:
        time_columns = TIMING_COLUMNS[1:]

    number_of_columns = min(2, len(time_columns))
    number_of_rows = int(np.ceil(len(time_columns) / number_of_columns))
    fig, axes = plt.subplots(
        number_of_rows,
        number_of_columns,
        figsize=figsize,
        squeeze=False,
    )

    fit_result_frames = []
    for ax, time_column in zip(axes.flat, time_columns):
        timing_label = TIMING_LABELS.get(
            time_column,
            time_column.replace("_", " "),
        )
        _, fit_results = plot_average_growth(
            data=data,
            x_column=x_column,
            x_label=x_label,
            title=f"Average {timing_label} Time by {x_label}",
            configurations=configurations,
            use_log_scale=use_log_scale,
            no_legend=True,
            ax=ax,
            time_column=time_column,
            fit_models=fit_models,
            return_fit_results=True,
        )
        fit_result_frames.append(fit_results)

    for unused_ax in axes.flat[len(time_columns):]:
        unused_ax.set_visible(False)

    handles, labels = axes.flat[0].get_legend_handles_labels()
    if handles:
        fig.legend(
            handles,
            labels,
            title="Configuration / fit",
            loc="center left",
            bbox_to_anchor=(0.84, 0.5),
        )

    fig.tight_layout(rect=(0, 0, 0.83, 1))
    plt.show()

    nonempty_results = [
        frame for frame in fit_result_frames if not frame.empty
    ]
    if nonempty_results:
        fit_results = pd.concat(nonempty_results, ignore_index=True)
    else:
        fit_results = pd.DataFrame()

    return fig, axes, fit_results




## Component timing examples

**plot_timing_components** creates a panel containing heuristic, planning, solution-extraction, and MEHR timing graphs. It returns **(figure, axes, fit_results)**.

The next cell shows component growth against horizon and community count. Add **fit_models=("linear", "exponential")** to either call when fitted curves are wanted.


In [ ]:
# The same component graphs can be generated for any problem-size column.
component_horizon_figure, _, _ = plot_timing_components(
    data=average_times,
    x_column="Horizon",
    x_label="Horizon",
)

component_communities_figure, _, _ = plot_timing_components(
    data=average_times,
    x_column="communities",
    x_label="Number of Communities",
)


In [ ]:
# Example: compare both models for one component and selected configurations.
# Change time_column/configurations to analyse another timing series.
_, mehr_fit_results = plot_average_growth(
    data=average_times,
    x_column="Horizon_x_communities",
    x_label="Horizon × number of communities",
    title="MEHR Time with Linear and Exponential Fits",
    configurations=["Act-Utilitarianism"],
    time_column="Mehr_time",
    fit_models=("exponential"),
    show_fit_metrics=True,
    return_fit_results=True,
)
mehr_fit_results.sort_values(["Config_name", "MSE"])


## Total-time growth plots

The following plots use **average_times** and retain the existing total-runtime analysis. Calls to **plot_average_growth** default to **time_column="Total_time"**.


In [ ]:
fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(16, 6),
    sharey=True,
)

plot_average_growth(
    data=average_times,
    x_column="Horizon",
    x_label="Horizon",
    title="Average Total Time by Horizon",
    use_log_scale=True,
    no_legend=True,
    ax=axes[0],
)

plot_average_growth(
    data=average_times,
    x_column="communities",
    x_label="Number of communities",
    title="Average Total Time by Number of Communities",
    use_log_scale=True,
    no_legend=True,
    ax=axes[1],
)

# Put the shared legend immediately beside the right-hand graph.
handles, labels = axes[1].get_legend_handles_labels()

axes[1].legend(
    handles,
    labels,
    title="Configuration",
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    borderaxespad=0,
)

fig.tight_layout()
plt.show()

In [ ]:
plot_average_growth(
    data=average_times,
    x_column="communities",
    x_label="Number of communities",
    title="Average Total Time by Number of Communities",
)


In [ ]:
plot_average_growth(
    data=average_times,
    x_column="Horizon_x_communities",
    x_label="Horizon × number of communities",
    title="Average Total Time by Combined Problem Size",
)

## Plot horizon against total time for different communities.

In [ ]:
configurations = average_times["Config_name"].drop_duplicates()

for configuration in configurations:
    configuration_data = average_times[
        average_times["Config_name"] == configuration
    ]

    fig, ax = plt.subplots(figsize=(9, 6))

    for number_of_communities, community_data in configuration_data.groupby(
        "communities"
    ):
        community_data = community_data.sort_values("Horizon")

        ax.plot(
            community_data["Horizon"],
            community_data["Average_total_time"],
            marker="o",
            linewidth=2,
            label=f"{number_of_communities:g} communities",
        )

    ax.set_xlabel("Horizon")
    ax.set_ylabel("Average total time (log scale)")
    ax.set_title(
        f"Average Total Time by Horizon and Communities\n"
        f"{configuration}"
    )
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Problem size")

    fig.tight_layout()
    plt.show()

In [ ]:
configurations = average_times["Config_name"].drop_duplicates()

for configuration in configurations:
    configuration_data = average_times[
        average_times["Config_name"] == configuration
    ]

    fig, ax = plt.subplots(figsize=(9, 6))

    # One line for each horizon
    for horizon, horizon_data in configuration_data.groupby("Horizon"):
        horizon_data = horizon_data.sort_values("communities")

        ax.plot(
            horizon_data["communities"],
            horizon_data["Average_total_time"],
            marker="o",
            linewidth=2,
            label=f"Horizon {horizon:g}",
        )

    ax.set_xlabel("Number of Communities")
    ax.set_ylabel("Average total time (log scale)")
    ax.set_title(
        f"Average Total Time by Communities and Horizon\n"
        f"{configuration}"
    )

    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Horizon")

    # Only show ticks for community counts that actually exist
    community_ticks = sorted(configuration_data["communities"].unique())
    ax.set_xticks(community_ticks)

    fig.tight_layout()
    plt.show()